#### Processing 0 - Creating the SpatialExperiment object

## Introduction

Here the SpatialExperiment obect to work with the ST data will be built from Space Ranger output.

In [1]:
setwd("/scratch_isilon/groups/singlecell/shared/projects/covid-pascual-reguant")

## Libraries

In [2]:
library(here)
library(glue)
library(stringr)
library(dplyr)
library(ggplot2)
library(SpatialExperiment)
library(scuttle)
library(HDF5Array)

here() starts at /scratch_isilon/groups/singlecell/shared/projects/covid-pascual-reguant


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: SingleCellExperiment

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: ‘matrixStats’


The following object is masked from ‘package:dplyr’:

    count



Attaching package: ‘MatrixGenerics’


The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,

## Setup

In [3]:
source(here("misc/paths.R"))

Rows: 20 Columns: 17
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (17): donor_id, study_id, disease_group, sex, age, sars-cov-2_antemortem...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


## Load samples

We will load individual samples as SpatialExperiment objects:

In [4]:
ids_ls <- sample_metadata %>%
    dplyr::filter(!str_detect(string = library_id, pattern = "\\*SP117_\\*")) %>%
    pull(donor_tissue)

spe_ls <- lapply(ids_ls, function(id_tissue) {
    id <- sample_metadata %>% 
        dplyr::filter(donor_tissue == id_tissue) %>% 
        pull(donor_id)
    
    spaceranger_dir <- here::here(glue::glue("{map_sp}/projects/{id}/jobs/{id_tissue}"))
    
    spe <- SpatialExperiment::read10xVisium(
        samples = spaceranger_dir,
        sample_id = id_tissue,
        type = "HDF5", 
        data = "filtered",  
        images = "hires",  
        load = TRUE  
    )
    
    return(spe)
})
names(spe_ls) <- ids_ls

After loading the samples, we can add the **metadata** for each spot:

In [5]:
spe_ls <- lapply(names(spe_ls), function(id_tissue) {
    spe <- spe_ls[[id_tissue]]
    
    cond_tissue <- sample_metadata %>%
        dplyr::filter(donor_tissue == id_tissue) %>%
        dplyr::pull(study_id_tissue)
    
    colData(spe)$cond_tissue <- cond_tissue
    colData(spe)$donor_tissue <- id_tissue
    
    cd <- as.data.frame(colData(spe)) %>%
        left_join(sample_metadata, by = c("cond_tissue" = "study_id_tissue")) %>%
        dplyr::select(-c(library_id, donor_tissue.y, donor_id)) %>%
        dplyr::rename(donor_tissue = donor_tissue.x)
    
    colData(spe) <- DataFrame(cd, row.names = colnames(spe))
    
    return(spe)
})
names(spe_ls) <- ids_ls

Now, we can finally **merge** all samples into one SpatialExperiment object:

In [6]:
spe <- do.call(cbind, spe_ls)
spe

class: SpatialExperiment 
dim: 36614 54984 
metadata(0):
assays(1): counts
rownames(36614): ENSG00000243485 ENSG00000237613 ... SCoV2_ORF10
  SCoV2_UTR3
rowData names(1): symbol
colnames(54984): AAACAAGTATCTCCCA-1 AAACACCAATAACTGC-1 ...
  TTGTTTCCATACAACT-1 TTGTTTGTGTAAATTC-1
colData names(21): in_tissue array_row ... sample_number tcr_id
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):
spatialCoords names(2) : pxl_col_in_fullres pxl_row_in_fullres
imgData names(4): sample_id image_id data scaleFactor

## Calculate QC metrics

Using the function `addPerCellQC()` we can add basic QC metrics for each spot, as:

 - **sum** = number of counts = Total UMI counts per spot
 - **detected** = number of features = Number of unique genes detected per spot

We will also calculate the percentage of **Mitochondrial**, **Ribosomal** and **SARS-CoV2** genes found in each spot.

We would first subset the object to keep only spots covered by the tissue sections. In our case, is it not necessary.

In [7]:
table(spe$in_tissue)


 TRUE 
54984 

In [8]:
is_mito <- grepl("^MT-", rowData(spe)$symbol)
table(is_mito)

is_mito
FALSE  TRUE 
36601    13 

In [9]:
is_ribo <- grepl("^RP[LS]", rowData(spe)$symbol)
table(is_ribo)

is_ribo
FALSE  TRUE 
36511   103 

In [10]:
is_cov <- grepl("^SCoV2", rowData(spe)$symbol)
table(is_cov)

is_cov
FALSE  TRUE 
36601    13 

In [11]:
spe <- addPerCellQC(spe, subsets = list(mito = is_mito, ribo = is_ribo, SCoV2 = is_cov))

In [12]:
summary(colData(spe)$subsets_ribo_percent)
summary(colData(spe)$subsets_mito_percent)
summary(colData(spe)$subsets_SCoV2_percent)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   6.891   9.188  10.920  13.962  50.607 

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   1.164   2.441   3.437   4.559  50.000 

    Min.  1st Qu.   Median     Mean  3rd Qu.     Max. 
 0.00000  0.00000  0.00000  0.03782  0.00000 49.66475 

In [13]:
head(colData(spe))

DataFrame with 6 rows and 33 columns
                   in_tissue array_row array_col   sample_id
                   <logical> <integer> <integer> <character>
AAACAAGTATCTCCCA-1      TRUE        50       102   S198-lung
AAACACCAATAACTGC-1      TRUE        59        19   S198-lung
AAACAGAGCGACTCCT-1      TRUE        14        94   S198-lung
AAACAGTGTTCCTGGG-1      TRUE        73        43   S198-lung
AAACATTTCCCGGATT-1      TRUE        61        97   S198-lung
AAACCCGAACGAAATC-1      TRUE        45       115   S198-lung
                           cond_tissue donor_tissue       study_id
                           <character>  <character>    <character>
AAACAAGTATCTCCCA-1 control_case_1-lung    S198-lung control_case_1
AAACACCAATAACTGC-1 control_case_1-lung    S198-lung control_case_1
AAACAGAGCGACTCCT-1 control_case_1-lung    S198-lung control_case_1
AAACAGTGTTCCTGGG-1 control_case_1-lung    S198-lung control_case_1
AAACATTTCCCGGATT-1 control_case_1-lung    S198-lung control_case_1
AAACCC

## Save data

Before saving the data to proceed with the QC, we will remove those genes that aren’t expressed in any of the spots overlaying the samples.

In [14]:
keep <- rowSums(counts(spe)) != 0
table(keep)

keep
FALSE  TRUE 
 6164 30450 

In [15]:
spe <- spe[keep, ]

In [16]:
saveHDF5SummarizedExperiment(spe, replace = TRUE, dir = "/home/groups/singlecell/cvicente/SP-COVID/visium/0-processing/out/0-QC_before_spe")

## Session Information

In [17]:
sessionInfo()

R version 4.4.3 (2025-02-28)
Platform: x86_64-conda-linux-gnu
Running under: CentOS Linux 7 (Core)

Matrix products: default
BLAS/LAPACK: /home/groups/singlecell/cvicente/miniconda3/envs/COVID/lib/libopenblasp-r0.3.29.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Madrid
tzcode source: system (glibc)

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] HDF5Array_1.34.0            rhdf5_2.50.2               
 [3] DelayedArray_0.32.0         SparseArray_1.6.2          
 [5] S4Arrays_1.6.0              abind_1.4-8                
 [7] Matrix_1.